# 02 - Dataset Creation

This notebook creates the processed datasets for all assets and horizons:
1. Load raw CSV files
2. Preprocess (clean, select features)
3. Time-based train/val/test split
4. Fit scaler on training data, transform all splits
5. Create sliding windows (X, y pairs)
6. Save as NumPy arrays + scaler pickle

Output: `data/processed/<category>/<asset>/<horizon>/`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from pathlib import Path

from src.data.windowing import create_dataset
from src.utils.config import ProjectConfig

config = ProjectConfig('..')
print(f'Categories: {config.get_categories()}')
print(f'Horizons: {config.get_horizons()}')
print(f'Window size: {config.dataset["window_size"]}')
print(f'Features: {config.dataset["features"]}')
print(f'Max samples (ENFORCED): {config.dataset["max_samples"]}')

## Create Datasets for All Assets and Horizons

In [ ]:
# Process all datasets
results = []

for category in config.get_categories():
    assets = config.get_assets_for_category(category)
    for asset in assets:
        for horizon in config.get_horizons():
            # Get dynamically paired window size
            window_size = config.get_window_size(horizon)
            
            print(f'\nProcessing: {category}/{asset["name"]}/h={horizon} (window={window_size})')
            
            raw_path = config.get_path('data_raw') / asset['file']
            output_dir = (config.get_path('data_processed') / category 
                         / asset['name'] / str(horizon))
            
            data = create_dataset(
                file_path=raw_path,
                output_dir=output_dir,
                window_size=window_size,
                horizon=horizon,
                features=config.dataset['features'],
                target=config.dataset['target'],
                train_ratio=config.dataset['split']['train'],
                val_ratio=config.dataset['split']['val'],
                test_ratio=config.dataset['split']['test'],
                scaler_type=config.dataset['scaler_type'],
                csv_columns=config.dataset.get('csv_columns'),
                datetime_format=config.dataset.get('datetime_format'),
                max_samples=config.dataset['max_samples'],
                force_recreate=True,  # Force recreate to apply window size changes
            )
            
            # Validate max_samples enforcement
            total = data['train_x'].shape[0] + data['val_x'].shape[0] + data['test_x'].shape[0]
            assert total <= config.dataset['max_samples'], (
                f"Dataset size {total} exceeds max_samples={config.dataset['max_samples']}"
            )
            
            results.append({
                'category': category,
                'asset': asset['name'],
                'horizon': horizon,
                'window_size': window_size,
                'train_x': data['train_x'].shape,
                'val_x': data['val_x'].shape,
                'test_x': data['test_x'].shape,
                'total_samples': total,
            })
            
            print(f'  Train: {data["train_x"].shape[0]}, Val: {data["val_x"].shape[0]}, '
                  f'Test: {data["test_x"].shape[0]}, Total: {total} (<= {config.dataset["max_samples"]} ✓)')
            
print(f'\n{"="*60}')
print(f'All datasets respect max_samples={config.dataset["max_samples"]} limit ✓')

## Summary of Created Datasets

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df['train_samples'] = results_df['train_x'].apply(lambda x: x[0])
results_df['val_samples'] = results_df['val_x'].apply(lambda x: x[0])
results_df['test_samples'] = results_df['test_x'].apply(lambda x: x[0])

print(f'\nTotal datasets created: {len(results_df)}')
print(f'Categories: {results_df["category"].nunique()}')
print(f'Assets: {results_df["asset"].nunique()}')
print(f'Horizons: {results_df["horizon"].nunique()}')
print(f'\nAll datasets respect max_samples={config.dataset["max_samples"]}:')
print(f'  Max total_samples: {results_df["total_samples"].max()}')
print(f'  Min total_samples: {results_df["total_samples"].min()}')
print(f'  Mean total_samples: {results_df["total_samples"].mean():.1f}')

results_df[['category', 'asset', 'horizon', 'train_samples', 'val_samples', 'test_samples', 'total_samples']]

In [ ]:
# Verify a sample dataset
import json

sample = results[0]
sample_dir = (config.get_path('data_processed') / sample['category'] 
             / sample['asset'] / str(sample['horizon']))

print(f"Sample directory: {sample_dir}")
print(f"Files: {sorted([f.name for f in sample_dir.iterdir()])}")

# Load and verify shapes
train_x = np.load(sample_dir / 'train_x.npy')
train_y = np.load(sample_dir / 'train_y.npy')
print(f"\ntrain_x shape: {train_x.shape} (samples, window_size, features)")
print(f"train_y shape: {train_y.shape} (samples, horizon)")
print(f"\ntrain_x sample (first 3 timesteps):\n{train_x[0, :3, :]}")
print(f"train_y sample: {train_y[0]}")

# Load and display metadata
metadata_path = sample_dir / 'split_metadata.json'
if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    print(f"\n{'='*60}")
    print("SPLIT METADATA")
    print(f"{'='*60}")
    print(f"Raw data range: {metadata['raw_start_datetime']} to {metadata['raw_end_datetime']}")
    print(f"Raw timesteps: {metadata['raw_total_timesteps']}")
    print(f"Window size: {metadata['window_size']}, Horizon: {metadata['horizon']}")
    print(f"Max samples enforced: {metadata['max_samples']}")
    print(f"\nTrain split: {metadata['splits']['train']['n_samples']} samples")
    print(f"  Date range: {metadata['splits']['train']['start_datetime']} to {metadata['splits']['train']['end_datetime']}")
    print(f"Val split: {metadata['splits']['val']['n_samples']} samples")
    print(f"  Date range: {metadata['splits']['val']['start_datetime']} to {metadata['splits']['val']['end_datetime']}")
    print(f"Test split: {metadata['splits']['test']['n_samples']} samples")
    print(f"  Date range: {metadata['splits']['test']['start_datetime']} to {metadata['splits']['test']['end_datetime']}")
    print(f"\nTotal windowed samples: {metadata['total_windowed_samples']}")

print('\nDataset creation complete. Proceed to 03_hpo.ipynb.')

## Split Metadata

Each dataset now includes a `split_metadata.json` file containing:
- **Global info**: Raw datetime range, features, window_size, horizon, max_samples
- **Per-split info**: Number of samples, start/end datetime, split ratio

This metadata enables:
- ✅ Reproducibility: Track exact datetime ranges used
- ✅ Validation: Verify windowing applied correctly
- ✅ Debugging: Identify temporal gaps or issues
- ✅ Analysis: Compare temporal coverage across assets

In [ ]:
# Display full metadata for one dataset
sample_metadata_path = (config.get_path('data_processed') / results[0]['category'] 
                       / results[0]['asset'] / str(results[0]['horizon']) / 'split_metadata.json')

if sample_metadata_path.exists():
    with open(sample_metadata_path, 'r') as f:
        sample_metadata = json.load(f)
    print("Full metadata structure:")
    print(json.dumps(sample_metadata, indent=2))